# Red flags del dataset ALeaseBERT para el Clause Analyzer

Las red flags no las produce el sistema: las anotó una persona en el dataset de Leivaditi, Rossi y
Kanoulas (2020), *A Benchmark for Lease Contract Review*. Este cuaderno las pasa a
`infra/json/red_flags.json`, que el servidor sirve en `red_flags/<doc_id>` y el Clause Analyzer
muestra en una tabla al final del panel cuando el contrato tiene alguna.

Cada red flag sale del `.ann.json` del contrato (clase `e_23`, su tipo en el campo `f_24`).
`paragraph_map.json`, que escribe `pool_to_docx.py`, dice en qué párrafo del DOCX quedó la línea
anotada. El artículo es el último título centrado del DOCX antes de ese párrafo.

Las claves son el id con que el sistema conoce cada documento: `root::` y el nombre del DOCX en
`infra/contracts_docx/` sin extensión, p. ej. `root::lease_contract_166`. Se escriben los 115
contratos con red flags, estén o no cargados en el sistema.

In [1]:
import json
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'infra').is_dir():
    ROOT = ROOT.parent
DATASET_DIR = ROOT.parent / 'datasets' / 'annotated_dataset'
POOL = DATASET_DIR / 'pool'
DOCX_DIR = DATASET_DIR / 'docx_red_flags'
OUT = ROOT / 'infra' / 'json' / 'red_flags.json'
RED_FLAG_CLASS, TYPE_FIELD = 'e_23', 'f_24'
HEADING_STYLE = 'BodyTextcenter'  # el estilo «Body Text.center» de la plantilla

W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'

def docx_paragraphs(path):
    """(estilo, texto) de cada párrafo del cuerpo, en orden. Se lee el XML directamente."""
    with zipfile.ZipFile(path) as z:
        body = ET.fromstring(z.read('word/document.xml')).find(f'{W}body')
    out = []
    for p in body.findall(f'{W}p'):
        style = p.find(f'{W}pPr/{W}pStyle')
        out.append((style.get(f'{W}val') if style is not None else None, ''.join(t.text or '' for t in p.iter(f'{W}t'))))
    return out

def article_of(paras, index):
    """El último título centrado antes del párrafo (o el propio párrafo si lo es)."""
    for i in range(index, -1, -1):
        if paras[i][0] == HEADING_STYLE:
            return paras[i][1]
    return ''

In [2]:
maps = json.loads((DOCX_DIR / 'paragraph_map.json').read_text())
contracts = {}
for name, m in sorted(maps.items()):
    ann = json.loads((POOL / f"{m['tagtog_id']}.ann.json").read_text())
    paras = docx_paragraphs(DOCX_DIR / f'{name}.docx')
    flags = []
    for e in ann['entities']:
        if e['classId'] != RED_FLAG_CLASS:
            continue
        where = m['parts'][e['part']]
        offset = where['offset'] + e['offsets'][0]['start']
        flags.append({
            'article': article_of(paras, where['paragraph']),
            'type': e['fields'].get(TYPE_FIELD, {}).get('value'),
            'text': e['offsets'][0]['text'],
            'docxParagraph': where['paragraph'],
            '_order': (where['paragraph'], offset),
        })
    flags.sort(key=lambda f: f['_order'])  # en el orden del contrato
    for f in flags:
        del f['_order']
    contracts[f'root::{name}'] = flags

payload = {
    'source': 'Leivaditi, Rossi & Kanoulas (2020), A Benchmark for Lease Contract Review. '
              'Figshare DOI 10.21942/uva.19732993, CC BY-NC 4.0.',
    'contracts': contracts,
}
OUT.write_text(json.dumps(payload, indent=1, ensure_ascii=False))

total = sum(len(v) for v in contracts.values())
print('contratos:', len(contracts), '| red flags:', total)
print('sin artículo (antes del primer título):', sum(1 for v in contracts.values() for f in v if not f['article']))
print('sin tipo:', sum(1 for v in contracts.values() for f in v if not f['type']))
print('guardado en', OUT.relative_to(ROOT))

contratos: 115 | red flags: 1252
sin artículo (antes del primer título): 20
sin tipo: 10
guardado en infra/json/red_flags.json


## Ejemplo: `lease_contract_201`

In [3]:
for f in contracts['root::lease_contract_201']:
    print(f"{f['article'][:40]:40s} | {f['type']:20s} | {f['text'][:70]}")

Article 2. Lease term                    | break_option         | Party A has right to terminate the contract and takes back the house i
Article 2. Lease term                    | break_option         | Party B transfers, subleases, lends, jointly operates, buys share or e
Article 2. Lease term                    | break_option         | Party B uses the house to take illegal activities, which damages the p
Article 2. Lease term                    | break_option         | Party B has the priority to lease the house after the contract expires
Article 3: Rent and rent paying agreemen | services_charges     | The water & electricity fees shall be settled separately.
Article 4: Repairing and decoration of h | landlord_repairs     | Party A takes charge of repairing of the house. Party A shall take the
Article 4: Repairing and decoration of h | landlord_repairs     | The repair scope and standard shall be performed by Urban Construction
Article 4: Repairing and decoration of h | landlord_re